# 09 — IBM-Ready Path

All previous notebooks used the **local Aer simulator** — no account needed.

This notebook shows what changes when you want to run on **real IBM quantum hardware**:
- Account setup and backend selection
- Transpilation for the target hardware topology
- Dealing with noise and error mitigation
- Interpreting results from noisy hardware

> **Note:** This notebook is designed to run in simulator mode by default.  
> The IBM hardware path is clearly marked and requires an IBM Quantum account.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import sys, pathlib
sys.path.insert(0, str(pathlib.Path.cwd().parent))

from helpers.notebook_style import setup_notebook
from helpers.plotting import plot_counts
setup_notebook()

from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator

IBM_MODE = False   # Set to True and configure credentials to use real hardware

def run_local(qc, shots=1024):
    sim = AerSimulator()
    job = sim.run(qc.decompose(), shots=shots)
    return job.result().get_counts()


## Step 1 — Circuit preparation

Prepare a clean circuit using the patterns from earlier notebooks.  
For IBM hardware, keep circuits **short** (depth matters for fidelity).


In [ ]:
# Bell state — a canonical two-qubit benchmark
qc = QuantumCircuit(2)
qc.h(0)
qc.cx(0, 1)
qc.measure_all()

print("Circuit to run:")
print(qc.draw(output="text"))


## Step 2 — Local simulator baseline

Always establish a simulator baseline **before** running on hardware.  
This tells you what the ideal (noiseless) result should look like.


In [ ]:
counts_ideal = run_local(qc, shots=4096)
print("Ideal simulator counts:", counts_ideal)

fig, ax = plt.subplots(figsize=(4, 3))
plot_counts(counts_ideal, title="Bell state — ideal simulator", ax=ax)
plt.tight_layout()
plt.show()


## Step 3 — Noisy simulation

Before touching real hardware, test with a **noise model** to get a realistic preview.
The Aer simulator can mimic the noise profile of real IBM devices.


In [ ]:
from qiskit_aer.noise import NoiseModel, depolarizing_error, thermal_relaxation_error
import qiskit_aer.noise as noise

# Simple depolarising noise model
noise_model = NoiseModel()
p_depol = 0.01   # 1% depolarising error on each gate
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(p_depol, 1), ["h", "rx", "ry", "rz"]
)
noise_model.add_all_qubit_quantum_error(
    depolarizing_error(p_depol * 10, 2), ["cx"]
)

noisy_sim = AerSimulator(noise_model=noise_model)
job_noisy = noisy_sim.run(qc.decompose(), shots=4096)
counts_noisy = job_noisy.result().get_counts()
print("Noisy simulation counts:", counts_noisy)

fig, axes = plt.subplots(1, 2, figsize=(8, 3))
plot_counts(counts_ideal, title="Ideal", ax=axes[0])
plot_counts(counts_noisy, title="With 1% depolarising noise", ax=axes[1])
plt.suptitle("Bell state: ideal vs. noisy", y=1.02)
plt.tight_layout()
plt.show()


## Step 4 — IBM hardware path (requires account)

```python
# ── IBM Quantum account setup ────────────────────────────────────────
from qiskit_ibm_runtime import QiskitRuntimeService, SamplerV2 as Sampler
from qiskit.transpiler.preset_passmanagers import generate_preset_pass_manager

# Load saved credentials (run once: QiskitRuntimeService.save_account(token="..."))
service = QiskitRuntimeService()

# Select the least-busy backend with enough qubits
backend = service.least_busy(operational=True, simulator=False, min_num_qubits=2)
print(f"Running on: {backend.name}")

# Transpile for the target hardware topology
pm = generate_preset_pass_manager(backend=backend, optimization_level=1)
qc_transpiled = pm.run(qc)

print("Transpiled circuit:")
print(qc_transpiled.draw(output="text", fold=-1))

# Run on hardware
sampler = Sampler(backend)
job = sampler.run([qc_transpiled], shots=4096)
result = job.result()
counts_hw = result[0].data.meas.get_counts()
print("Hardware counts:", counts_hw)
```

> Replace `IBM_MODE = False` with `IBM_MODE = True` at the top of this notebook  
> once you have configured your IBM Quantum credentials.


## Step 5 — Reading hardware results

On real hardware you will see:
- **Reduced correlation**: the 00/11 counts will be less clean due to gate errors
- **Spurious states**: small contributions from 01 and 10 due to readout errors
- **Run-to-run variation**: results differ slightly between submissions

Error mitigation (e.g. ZNE, M3) can partially correct for these effects.  
`rqm-qiskit` will provide built-in mitigation helpers in a future release.


## Summary

| Step | Tool | Mode |
|---|---|---|
| Circuit design | `rqm-core` + `rqm-qiskit` | Always |
| Ideal baseline | Aer simulator | Always |
| Noise preview | Aer + NoiseModel | Before hardware |
| Hardware run | `qiskit-ibm-runtime` | IBM account required |
| Error mitigation | TBD in `rqm-qiskit` | Future |

The simulator-first approach minimises hardware queue time and cost while letting you validate your logic locally.

---

**You have completed the rqm-notebooks learning path!**  
Return to [00_welcome_and_repo_map.ipynb](00_welcome_and_repo_map.ipynb) for a summary of what you have learned.
